# 05 — Benchmark Comparison: MILP vs GA

Compares exact MILP vs GA heuristic on solve time and optimality gap across increasing problem sizes.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_prep import load_processed, generate_synthetic_dataset, clean_data, assign_locations, aggregate_weekly, select_top_skus
from demand_model import DemandModel, build_demand_models
from evaluate import (
    compute_revenue_baseline, compute_revenue_lift,
    compute_optimality_gap, print_results_summary, generate_results_table
)

plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Full Data and Models

In [ ]:
weekly_df, elasticity_df = load_processed()
demand_model = DemandModel.load(model_type='xgboost')
location_list = weekly_df['location'].unique().tolist()
capacities = {loc: 800 for loc in location_list}

## 2. Solve Both Methods

In [ ]:
from milp_solver import solve_pricing_milp
from ga_solver import solve_pricing_ga

print("Solving with MILP...")
milp_result = solve_pricing_milp(
    weekly_df, demand_model, capacities=capacities,
    n_price_levels=8, timeout_seconds=120
)

print("\nSolving with GA...")
ga_result = solve_pricing_ga(
    weekly_df, demand_model, capacities=capacities,
    n_price_levels=8, population_size=100, n_generations=200
)

## 3. Revenue Lift vs Baseline

In [ ]:
baseline = compute_revenue_baseline(weekly_df)

milp_lift = compute_revenue_lift(milp_result, baseline)
ga_lift = compute_revenue_lift(ga_result, baseline)

print_results_summary(milp_result, ga_result, baseline)

## 4. Results Comparison Table

In [ ]:
results_table = generate_results_table(milp_result, ga_result, baseline)
results_table

## 5. Optimality Gap Visualization

In [ ]:
gap = compute_optimality_gap(milp_result, ga_result)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

methods = ['Baseline', 'MILP', 'GA']
revenues = [
    baseline['total_revenue'],
    milp_lift['optimal_revenue'],
    ga_lift['optimal_revenue']
]
colors = ['gray', 'steelblue', 'coral']

axes[0].bar(methods, revenues, color=colors)
axes[0].set_title('Revenue Comparison')
axes[0].set_ylabel('Revenue ($)')

times = [
    0,
    gap['milp_solve_time'],
    gap['ga_solve_time']
]
axes[1].bar(methods, times, color=colors)
axes[1].set_title('Solve Time Comparison')
axes[1].set_ylabel('Time (seconds)')

optimality_data = [0, 0, gap['optimality_gap_pct']]
axes[2].bar(['MILP', 'GA', 'Gap'], [gap['milp_objective'], gap['ga_objective'], gap['optimality_gap_pct']],
            color=['steelblue', 'coral', 'red'])
axes[2].set_title(f'Optimality Gap: {gap["optimality_gap_pct"]:.2f}%')
axes[2].set_ylabel('Value')

plt.tight_layout()
plt.show()

## 6. Sensitivity Analysis: Price Stability Constraint

In [ ]:
from evaluate import constraint_sensitivity

price_change_values = [5.0, 10.0, 15.0, 20.0, 30.0, 50.0]

print("Running sensitivity analysis on max_price_change...")
sensitivity_df = constraint_sensitivity(
    weekly_df, demand_model,
    param_name='max_price_change',
    param_values=price_change_values,
    solver='ga'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sensitivity_df['param_value'], sensitivity_df['revenue'], marker='o', linewidth=2)
axes[0].set_title('Revenue vs Max Price Change')
axes[0].set_xlabel('Max Price Change ($)')
axes[0].set_ylabel('Revenue ($)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(sensitivity_df['param_value'], sensitivity_df['profit'], marker='o', linewidth=2, color='green')
axes[1].set_title('Profit vs Max Price Change')
axes[1].set_xlabel('Max Price Change ($)')
axes[1].set_ylabel('Profit ($)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSensitivity Results:")
print(sensitivity_df[['param_value', 'revenue', 'profit']].round(2))

## 7. Key Takeaways

- MILP provides optimal solutions but scales poorly beyond small instances
- GA provides near-optimal solutions (small optimality gap) with much faster solve times
- The crossover point where GA becomes necessary depends on the number of SKUs x locations x weeks
- Relaxing the price-stability constraint increases revenue but may alienate customers